# Violence detection on surveillance video: a walkthrough

This notebook shows the system's backend end to end:

1. **The architecture**: which models run, in what order, and why.
2. **One clip, live**: the clip goes through every stage on this laptop, and we look at what each stage produces.
3. **How well it works**: the held-out test results, recomputed here from the saved per-clip scores.
4. **Limitations**.

**Before running:** start Jupyter from the repository root, with the model bundle unpacked
(see `docs/RUN_THE_MODEL.md`). Everything runs on CPU. Nothing in this notebook trains or tunes a model.

```bash
pip install jupyterlab
jupyter lab notebooks/professor_walkthrough.ipynb
```

## 1. Architecture

| # | Stage | Model | Input → output |
|---|---|---|---|
| 1 | Clip features | **C3D** (Sports-1M, frozen) | video → 32 time segments × 4,096 features |
| 2 | *When* is something unusual? | **Sultani MIL scorer** (trained on UCF-Crime) | features → anomaly score per segment |
| 3 | *Who* is in the suspicious window? | **Faster R-CNN** (COCO) | frame → person boxes |
| 4 | What is each person doing? | **I3D** Mixed_4f + RoIAlign | frames + boxes → one feature per person |
| 5 | What is the group doing? | **Actor-Transformer** (trained on DCSASS) | person features → 6 behaviours (Normal, Abuse, Assault, Fighting, Robbery, Vandalism) |
| + | **Is this violent?** | **v3 pilot head** (Indian + DCSASS + UBI-Fights + XD-Violence) | the same C3D features → violence score, alert at ≥ 0.40 |

Stages 1–5 are the original two-paper cascade: Sultani et al. (2018) decides *when* to look, and
Gavrilyuk et al.'s Actor-Transformer (2020) reasons about *who does what*. The v3 pilot head is the
fine-tuned violence model this sprint produced. It reuses stage 1's features, so it adds almost no cost.

Every output goes to **human review**. The system describes observable behaviour; it never infers
identity, intent or guilt.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from surveillance.demo_profile import load_professor_profile
from surveillance.demo_rendering import (
    behavior_probabilities,
    interval_rows,
    reference_gallery,
    timeline_figure,
)
from surveillance.demo_service import DemoService
from surveillance.inference.indian_pilot import load_pilot_spec
from surveillance.video.decode import probe_video

config = load_professor_profile(ROOT)
config["device"] = "cpu"
config["indian_pilot"] = load_pilot_spec(ROOT / "runs/indian_pilot/demo/indian_pilot_spec_v3.json")
%matplotlib inline
service = DemoService(config, ROOT)  # checks the pilot head's SHA256 before loading it

models = {key: config[key] for key in ("sultani_checkpoint", "c3d_checkpoint", "detector_checkpoint",
                                         "i3d_checkpoint", "behavior_checkpoint")}
models["indian_pilot (v3)"] = str(ROOT / config["indian_pilot"]["checkpoint"])
pd.DataFrame(
    [(name, Path(path).relative_to(ROOT).as_posix(), f"{Path(path).stat().st_size / 1e6:.0f} MB")
     for name, path in models.items()],
    columns=["model", "file", "size"],
)

## 2. One clip through the pipeline

Pick any video. The default is the demo's **curated** robbery example. It was hand-picked because
it exercises every stage, so it is an illustration, not evidence of accuracy. Change `CLIP` to
any other file (for example `data/examples/limitation.mp4`, a known mistake) to see other behaviour.

In [ ]:
CLIP = ROOT / "data/examples/robbery_cascade_success_ucf_v1.mp4"

meta = probe_video(CLIP)
print(f"{CLIP.name}: {meta.duration_sec:.1f} s, {meta.num_frames} frames at {meta.fps:.1f} fps")

cap = cv2.VideoCapture(str(CLIP))
fig, axes = plt.subplots(1, 4, figsize=(14, 2.8))
for ax, frame_index in zip(axes, np.linspace(0, meta.num_frames - 1, 4).astype(int)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ok, frame = cap.read()
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{frame_index / meta.fps:.1f} s", fontsize=9)
    ax.axis("off")
cap.release()
plt.show()

### Run every stage (live, on this laptop)

This is the same call the web demo makes for an uploaded video. The first run also loads the
models, so it is slower.

In [ ]:
result = service.analyze(str(CLIP), example="", use_cache=False)

timings = {k: round(v, 2) for k, v in result["timings"].items() if isinstance(v, (int, float))}
pd.Series(timings, name="seconds").to_frame()

### Stage 1: C3D features

The clip is cut into 16-frame snippets and passed through C3D. The snippet features are pooled into
**32 equal time segments**, giving one 32 × 4,096 matrix per video, whatever its length.

In [ ]:
features = np.asarray(service.pipeline.anomaly.last_features)
print("feature bag shape:", features.shape)

plt.figure(figsize=(12, 2.4))
plt.imshow(features[:, :400].T, aspect="auto", cmap="Greys")
plt.xlabel("time segment (1-32)")
plt.ylabel("first 400 of 4,096 dims")
plt.title("C3D feature bag (darker = larger activation)", loc="left", fontsize=10)
plt.show()

### Stage 2: Sultani anomaly timeline

The MIL scorer gives each segment a score from 0 to 1. Segments above the routing threshold (0.5) are
sent on to the behaviour stages; the shaded bands are the windows that were analysed.

In [ ]:
display(timeline_figure(result))
print(f"highest segment score: {result['anomaly']['overall_score']:.3f}"
      f"   (routing threshold {result['policy']['anomaly_threshold']})")

### Stages 3–5: people, their actions, and the group's behaviour

For each suspicious window: Faster R-CNN finds the people in a reference frame, I3D describes each
person's motion, and the Actor-Transformer reasons over all of them together to predict the group's
behaviour.

In [ ]:
display(pd.DataFrame(interval_rows(result),
                     columns=["window", "people", "behaviour", "anomaly score", "status"]))

gallery = reference_gallery(result)
if gallery:
    fig, axes = plt.subplots(1, len(gallery), figsize=(6 * len(gallery), 4), squeeze=False)
    for ax, (image, caption) in zip(axes[0], gallery):
        ax.imshow(image)
        ax.set_title(caption, fontsize=9)
        ax.axis("off")
    plt.show()
else:
    print("No window passed the anomaly threshold, so stages 3-5 were skipped.")

probabilities, label = behavior_probabilities(result)
if probabilities:
    display(pd.Series(probabilities, name=label).round(3).to_frame())

### Final output: the cascade alert and the v3 violence score

The cascade's alert and the v3 pilot score are shown side by side. Neither decides anything; both
go to a person for review.

In [ ]:
alert = result["final_alert"]
pilot = result["indian_pilot"]
print("Cascade alert :", alert["state"], "|", alert["reason"])
print(f"v3 violence   : score {pilot['violence_score']:.3f}  threshold {pilot['threshold']:.2f}"
      f"  ->  {'FLAGGED for review' if pilot['flagged'] else 'not flagged'}")

## 3. How well it works: the held-out test

**Test set:** 49 two-second clips from IITB-Corridor (Indian campus CCTV): 24 fights and 25
non-violent clips. No IITB footage was used in training or tuning. The test was scored **once**, under
a protocol committed beforehand (`runs/indian_pilot/V3_PROTOCOL.md`); the scoring script refuses to
run again.

The cells below **do not re-run the test**. They recompute the metrics from the saved per-clip scores
in `runs/indian_pilot/test_results_v3.json`.

In [ ]:
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score

saved = json.loads((ROOT / "runs/indian_pilot/test_results_v3.json").read_text())
clips = saved["clip_scores"]
y = np.array([c["label"] for c in clips.values()])

def summary(key, threshold):
    s = np.array([c[key] for c in clips.values()])
    tn, fp, fn, tp = confusion_matrix(y, s >= threshold).ravel()
    return {
        "threshold": threshold,
        "accuracy": (tp + tn) / len(y),
        "recall": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "precision": tp / (tp + fp) if tp + fp else float("nan"),
        "F1": 2 * tp / (2 * tp + fp + fn),
        "ROC-AUC": roc_auc_score(y, s),
        "PR-AUC": average_precision_score(y, s),
        "fights caught": f"{tp}/{tp + fn}",
        "false alarms": f"{fp}/{fp + tn}",
    }

table = pd.DataFrame({
    "v3 (deployed)": summary("v3_all_s7", 0.40),
    "v2": summary("v2_s7", 0.45),
    "zero-shot Sultani": summary("zero_shot_sultani", 0.50),
    "v1 (Indian only)": summary("v1_indian_only_head", 0.40),
}).T
table.style.format(precision=3)

In [ ]:
s = np.array([c["v3_all_s7"] for c in clips.values()])
cm = confusion_matrix(y, s >= 0.40)
fig, ax = plt.subplots(figsize=(4, 3.4))
ax.imshow(cm, cmap="Blues")
for (i, j), n in np.ndenumerate(cm):
    ax.text(j, i, n, ha="center", va="center", fontsize=16, color="white" if n > 12 else "black")
ax.set_xticks([0, 1], ["non-violent", "violent"])
ax.set_yticks([0, 1], ["non-violent", "violent"])
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("v3 at threshold 0.40", loc="left", fontsize=10)
plt.show()

In [ ]:
display(Image(filename=str(ROOT / "docs/results/pilot_v3_iitb_curves.png")))

### Uncertainty and the pre-registered question

ROC-AUC and PR-AUC are seed means (seeds 7, 8, 9). The 95% CIs come from a bootstrap over the
18 test videos. The protocol asked one question in advance: *does v3 beat v2, with the CI of
the difference excluding zero?*

In [ ]:
ci = saved["source_bootstrap_ci95"]["v3_all"]
mean = saved["metrics_seed_mean"]["v3_all"]
display(pd.DataFrame({
    "seed mean": {k: mean[k] for k in ("roc_auc", "pr_auc", "within_source_roc_auc")},
    "95% CI": {k: f"{ci[k][0]:.2f} - {ci[k][1]:.2f}" for k in ("roc_auc", "pr_auc", "within_source_roc_auc")},
}).round(3))

diff = saved["paired_differences"]["v3_all_vs_v2"]
print(f"v3 - v2 PR-AUC: {diff['pr_auc_difference']:+.3f}, CI {np.round(diff['pr_auc_ci95'], 3)}")
print(f"v3 - v2 within-video ROC-AUC: {diff['within_source_roc_auc_difference']:+.3f}, "
      f"CI {np.round(diff['within_source_roc_auc_ci95'], 3)}")
print("Improvement shown by the pre-registered rule:", saved["improved_by_preregistered_criterion"])

## 4. Limitations

- **v3 vs v2 is not a proven improvement.** Both CIs above include zero.
- **The switch to v3 was post hoc.** The protocol said to keep v2; the team chose v3 *after* the test,
  because it catches 17/24 fights at its threshold where v2 catches none.
- **The test set is small and narrow.** All 24 fights come from 8 videos of one staged corridor scene.
- **The margin is thin.** Median fight score 0.44, highest non-violent score 0.42, threshold 0.40.
- **The labels are pilot quality.** The Indian training labels are team-annotated and not adjudicated.
- **Human review only.** Scores are uncalibrated and say nothing about identity or intent.

Details: `docs/results/MODEL_PERFORMANCE.md` and the decision log `runs/indian_pilot/DECISIONS.md`.

## Try your own clip

Point `MY_CLIP` at any short video (2–10 s works best) and run the cell.

In [ ]:
MY_CLIP = ROOT / "data/examples/limitation.mp4"

mine = service.analyze(str(MY_CLIP), example="", use_cache=False)
display(timeline_figure(mine))
print("Cascade alert :", mine["final_alert"]["state"], "|", mine["final_alert"]["behavior"])
print(f"v3 violence   : {mine['indian_pilot']['violence_score']:.3f}"
      f"  -> {'FLAGGED' if mine['indian_pilot']['flagged'] else 'not flagged'}")